# DWBI Week 5 — The ETL Process

**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A** · Superior University

PakFreight's Week-2 merge rebuilt as a pipeline that reruns: 37 rows land, 35 append, 34 load — 578,650 kg and PKR 4,808,000, exactly. Every number below is computed from `data/source/`, `data/dims/` or `data/night2/` in the notebook itself; nothing is typed in as a constant except the assertion target it is checked against.

## T2 — Stage ownership (12 marks)

Twelve activities, each owned by Extract, Transform or Load — or marked
**Violation**, meaning one stage is doing another stage's job, or a job no
stage may do at all. The three "must nots" are the point: extract must not
clean, transform must not write to the source, load must not decide anything.

In [1]:
# T2 grounding: the two facts the verdicts lean on, read from the real files
import pandas as pd
city = pd.read_csv("data/dims/DimCity.csv")
alias = pd.read_csv("data/alias_lookup.csv")
print("DimCity rows:", len(city), "| alias spellings:", len(alias),
      "| distinct keys:", alias.CityKey.nunique())
print("Port Qasim ->", alias.loc[alias.RawSpelling == "Port Qasim", "CityName"].iloc[0])
print("Pipri Marshalling Yard ->", alias.loc[alias.RawSpelling == "Pipri Marshalling Yard", "CityName"].iloc[0])
assert len(alias) == 23 and alias.CityKey.nunique() == 14

DimCity rows: 15 | alias spellings: 23 | distinct keys: 14
Port Qasim -> Karachi
Pipri Marshalling Yard -> Karachi


**Analysis.** The alias map really does resolve 23 spellings to 14 keys, so activity 7 ('Port Qasim means Karachi') is a real conforming decision with a lookup behind it, and activity 8's Unknown routing is a real branch of the pipeline rather than a hypothetical. The verdict table below is therefore argued against a lookup that exists, not an invented one.

In [2]:
# T2 verdict counts, computed from the verdict table itself
import pandas as pd
acts = pd.DataFrame([
    (1, "Open a connection to the rail workbook and read the named sheet", "Extract"),
    (2, "Strip the trailing space off a status value while reading the file in", "Violation"),
    (3, "Write the row count of each file into a run log", "Extract"),
    (4, "Promote row 3 of the rail sheet to be the header row", "Transform"),
    (5, "Correct the spelling of 'Pipri Marshalling Yard' in the railway's own database", "Violation"),
    (6, "Multiply the rail weight column by 1,000", "Transform"),
    (7, "Decide that 'Port Qasim' means Karachi", "Transform"),
    (8, "Route a city the alias map does not recognise to CityKey -1", "Transform"),
    (9, "Insert the four independent dimensions before FactShipment", "Load"),
    (10, "Work out what to do about a blank status value while writing the fact rows", "Violation"),
    (11, "Wrap the six table writes so that either all of them land or none do", "Load"),
    (12, "Publish the staging table to the sales team so they can get at the data sooner", "Violation"),
], columns=["n", "activity", "verdict"])
print(acts.verdict.value_counts().to_string())
assert (acts.verdict == "Extract").sum() == 2 and (acts.verdict == "Transform").sum() == 4
assert (acts.verdict == "Load").sum() == 2 and (acts.verdict == "Violation").sum() == 4

verdict
Violation    4
Transform    4
Extract      2
Load         2


**Analysis.** The self-check passes: Extract 2, Transform 4, Load 2, Violation 4. Transform owning the most work is not an accident of how the list was written — cleaning, conforming and keying are the bulk of what an ETL pipeline actually does, while extract and load are deliberately thin: one reads faithfully, the other writes what it is given.

| # | Activity | Verdict | Why — and, for violations, which rule it breaks |
|---|---|---|---|
| 1 | Open a connection to the rail workbook and read the named sheet | Extract | Reading is extract's whole job; nothing is changed, only located and opened. |
| 2 | Strip the trailing space off a status value while reading the file in | **Violation** | Extract must not clean. The moment the read "helpfully" trims, there is no untouched copy left to prove the source against, and a source problem becomes indistinguishable from your own edit. |
| 3 | Write the row count of each file into a run log | Extract | The log is extract's receipt: 37 rows landed, junk included. It proves the landing was faithful and gives every later stage a number to reconcile against. |
| 4 | Promote row 3 of the rail sheet to be the header row | Transform | Shaping is transform work: the file's layout is being restructured into a table, which is exactly what the staging layer exists to do. |
| 5 | Correct the spelling of 'Pipri Marshalling Yard' in the railway's own operational database | **Violation** | Transform must not write to the source. Fixing the source destroys the evidence, breaks the next team's extract, and turns a warehouse job into an unauthorised edit of an operational system. |
| 6 | Multiply the rail weight column by 1,000 | Transform | A unit conversion is a transform, done in staging, before anything is combined — which is why check 2 can read the rail subtotal at 295,250 kg before the append. |
| 7 | Decide that 'Port Qasim' means Karachi | Transform | Conforming is transform work: 23 spellings become 14 keys through the alias lookup, in staging, without touching any source file. |
| 8 | Route a city the alias map does not recognise to CityKey −1 | Transform | Key assignment, including the Unknown branch, is a transform decision — and because it is made here, the row is counted (night 1: 0, night 2: 1) instead of vanishing. |
| 9 | Insert the four independent dimensions before FactShipment | Load | Writing finished tables into the model, in dependency order, is load's job; the order matters because DimRoute and the fact both need city keys to exist first. |
| 10 | Work out what to do about a blank status value while writing the fact rows | **Violation** | Load must not decide anything. A decision invented at write time is unreviewed, unlogged and unrepeatable — it belongs in transform, as a named rule, before the load ever runs. |
| 11 | Wrap the six table writes so that either all of them land or none do | Load | Transactional integrity of the write itself is load's responsibility: a half-loaded model is worse than a stale one, because it reconciles against nothing. |
| 12 | Publish the staging table to the sales team so they can get at the data sooner | **Violation** | Staging is never user-facing (see paragraph below). Shipping the working surface means users build reports on uncleaned, unkeyed data that will move under them. |

### Why staging is never user-facing (the 4-mark paragraph)

Staging is a workshop floor, not a shop window: its tables carry raw spellings, unconverted tons,
unresolved keys and rows that will be dropped or rerouted before they are fit to report on, so any
report built on them is built on numbers that are *going* to change. Publishing them also destroys
the blame boundary that the three stage names exist to protect — once users query staging, a
staging quirk becomes a "warehouse error" that no stage owns. Power Query enforces the boundary
with a single setting: **Enable load, off** (right-click the query → uncheck it) on `Marine`,
`Rail`, `Road` and `AliasLookup`, which keeps them running on refresh and feeding the queries
below while producing no table in the model at all. Six tables in the model, ten queries behind it —
and the four staging queries are the four the users never see. In pandas the same boundary is a
convention rather than a setting, which is why the lab notebook states explicitly which frames are
intermediate and which are published.